In [0]:
%sh pip install sodapy

In [0]:
from sodapy import Socrata
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import json
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType

APP_TOKEN = dbutils.secrets.get("biz-survival", "nyc_app_token")

try:
    df_existing = spark.table("data_515.default.`311_service_requests`")
    last_created_date_str = df_existing.agg(F.max('created_date')).collect()[0][0]
    last_created_date = last_created_date_str.strftime("%Y-%m-%dT00:00:00.000")
    columns_str = ','.join(df_existing.columns) if isinstance(df_existing.columns, list) else df_existing.columns
except:
    last_created_date = "01/01/2026"
    column_set = "*"

client = Socrata("data.cityofnewyork.us", APP_TOKEN)

soql_query = f"SELECT {columns_str} WHERE created_date > '{last_created_date}' limit 500000"
results = client.get("erm2-nwe9", query=soql_query)

In [0]:
last_created_date

In [0]:
soql_query

In [0]:
df_existing = df_existing.withColumnsRenamed({
    'Unique Key': 'unique_key',
    'Created Date': 'created_date',
    'Closed Date': 'closed_date',
    'Agency': 'agency',
    'Agency Name': 'agency_name',
    'Problem (formerly Complaint Type)': 'complaint_type',
    'Problem Detail (formerly Descriptor)': 'descriptor',
    'Additional Details': 'additional_details',
    'Location Type': 'location_type',
    'Incident Zip': 'incident_zip',
    'Incident Address': 'incident_address',
    'Street Name': 'street_name',
    'Cross Street 1': 'cross_street_1',
    'Cross Street 2': 'cross_street_2',
    'Intersection Street 1': 'intersection_street_1',
    'Intersection Street 2': 'intersection_street_2',
    'Address Type': 'address_type',
    'City': 'city',
    'Landmark': 'landmark',
    'Facility Type': 'facility_type',
    'Status': 'status',
    'Due Date': 'due_date',
    'Resolution Description': 'resolution_description',
    'Resolution Action Updated Date': 'resolution_action_updated_date',
    'Community Board': 'community_board',
    'Council District': 'council_district',
    'Police Precinct': 'police_precinct',
    'BBL': 'bbl',
    'Borough': 'borough',
    'X Coordinate (State Plane)': 'x_coordinate_state_plane',
    'Y Coordinate (State Plane)': 'y_coordinate_state_plane',
    'Open Data Channel Type': 'open_data_channel_type',
    'Park Facility Name': 'park_facility_name',
    'Park Borough': 'park_borough',
    'Vehicle Type': 'vehicle_type',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Location': 'location'
})

In [0]:
df_new_data.display()

In [0]:
df_existing.display()

In [0]:
df_csv = spark.read.option("header", "true").csv("/Volumes/data_515/default/biz_survival/311_Service_Requests_from_2020_to_Present_20260222.csv")
df_csv_lower = df_csv.toDF(*[col.lower().replace(" ", "_") for col in df_csv.columns])\
    .withColumnRenamed("additional_details", "descriptor_2")\
    .withColumnRenamed("problem_(formerly_complaint_type)", "complaint_type")\
    .withColumnRenamed("problem_detail_(formerly_descriptor)", "descriptor")\
    .withColumnRenamed("x_coordinate_(state_plane)", "x_coordinate_state_plane")\
    .withColumnRenamed("y_coordinate_(state_plane)", "y_coordinate_state_plane")\
    .withColumn(
        "created_date",
        F.expr("try_to_timestamp(created_date, 'MM/dd/yyyy hh:mm:ss a')")
    ).withColumn(
        "closed_date",
        F.expr("try_to_timestamp(closed_date, 'MM/dd/yyyy hh:mm:ss a')")
    ).withColumn(
        "resolution_action_updated_date",
        F.expr("try_to_timestamp(resolution_action_updated_date, 'MM/dd/yyyy hh:mm:ss a')")
    )


In [0]:
df_csv_lower.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("data_515.default.311_service_requests")

In [0]:
if results:
    sample_record = results[0] if results else {}
    nyc_311_schema = StructType([StructField(k, StringType(), True) for k in sample_record.keys()])
    df_new_data = spark.createDataFrame(results, schema=nyc_311_schema)
    df_new_data = df_new_data.withColumn("created_date", F.to_timestamp("created_date"))\
                             .withColumn("closed_date", F.to_timestamp("closed_date"))\
                             .withColumn("resolution_action_updated_date", F.to_timestamp("resolution_action_updated_date"))
df_new_data.display()

In [0]:
df_new_data.count()

In [0]:
df_new_data.selectExpr("MIN(created_date)", "MAX(created_date)").display()

In [0]:
DeltaTable.forPath(spark, "data_515.default.311_service_requests") \
    .alias("target") \
    .merge(
        df_new_data.alias("source"),
        "target.unique_key = source.unique_key"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()
print("Merge completed")

In [0]:
from sodapy import Socrata
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import json
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType

APP_TOKEN = dbutils.secrets.get("biz-survival", "nyc_app_token")
lake_table_path = "data_515.default.nyc_311_service_requests"

try:
    df_existing = spark.table("data_515.default.`311_service_requests`")
    last_created_date_str = df_existing.agg(F.max('created_date')).collect()[0][0]
    last_created_date = last_created_date_str.strftime("%Y-%m-%dT00:00:00.000")
    columns_str = ','.join(df_existing.columns) if isinstance(df_existing.columns, list) else df_existing.columns
except:
    # Default to the start of the current month
    last_created_date = datetime.today().replace(day=1).strftime("%Y-%m-%dT00:00:00.000")
    column_set = "*"

print("Last Created Date: ", last_created_date)

client = Socrata("data.cityofnewyork.us", APP_TOKEN)
soql_query = f"SELECT {columns_str} WHERE created_date > '{last_created_date}' limit 500000"
results = client.get("erm2-nwe9", query=soql_query)

if results:
    sample_record = results[0] if results else {}
    nyc_311_schema = StructType([StructField(k, StringType(), True) for k in sample_record.keys()])
    df_new_data = spark.createDataFrame(results, schema=nyc_311_schema)
    df_new_data = df_new_data.withColumn("created_date", F.to_timestamp("created_date"))\
                             .withColumn("closed_date", F.to_timestamp("closed_date"))\
                             .withColumn("resolution_action_updated_date", F.to_timestamp("resolution_action_updated_date"))
    print("New Data Count:", df_new_data.count())
    DeltaTable.forPath(spark, "data_515.default.311_service_requests") \
    .alias("target") \
    .merge(
        df_new_data.alias("source"),
        "target.unique_key = source.unique_key"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()
    print("Merge completed")
    
else:
    print("No new data")


In [0]:
start_of_month = datetime.today().replace(day=1).strftime("%Y-%m-%dT00:00:00.000")
start_of_month

In [0]:
last_created_date